# 国医大师孙光荣中医知识图谱 · RAG 智能问答

**Sun Guangrong TCM Knowledge Graph — Graph-RAG Assistant**

> 研发：**孙光荣大师弟子田建辉团队** 联合 **医哲未来人工智能研究院（IMPF-AI Institute）**

在 Colab 里一键跑起一个**基于知识图谱检索增强（GraphRAG）的中医问答服务**，
并生成一个**公网可访问的多轮对话链接**（`*.gradio.live`）。

| | |
|---|---|
| 知识底座 | 19,343 实体 · 25,713 关系 · 8 部文献 · 31 类实体 · 44 类关系 |
| 检索 | 实体索引 + 原文佐证索引双路召回 → 1 跳图扩展 → 关系类型轮转采样 |
| 生成 | **MiniMax**（默认 `MiniMax-M3`，国内站）或 **Poe**（默认 `claude-sonnet-5`） |
| 默认检索档位 | 种子实体 35 · 关系 120 · 上下文 9,900 字（可在启动格改，也可整体隐藏面板） |
| 可核验 | 右侧面板实时展示本轮实际检索到的子图 / 实体 / 三元组 / 原文佐证 / 送入模型的上下文 |

**依次运行下面每个单元格即可**（约 2 分钟）。第 6 格是「RAG 知识图谱数据」预览，
不配 API Key 也能单独用；第 8 格产出公网链接。

---
> 本应用为中医药学术研究与教学参考工具，内容由文献自动抽取并经抽样审校，
> **不能替代执业医师的诊断与处方**，请勿据此自行用药。


## 1　安装依赖

In [ ]:
#@title 安装依赖（约 30 秒）
!pip -q install "gradio>=4.44" requests numpy

import gradio, numpy, requests
print("gradio", gradio.__version__, "| numpy", numpy.__version__)

## 2　取得图谱数据

按 `已在本地` → `GitHub 直链` → `手动上传` 的顺序尝试。
仓库为私有时前两步会失败，届时会弹出上传框，
上传 `graph.json` **或**原始的 `sunguangrong_kg_v2_explorer.html` 都可以（后者会自动抽取）。

In [ ]:
#@title 下载 / 上传图谱数据
import os, sys, subprocess

GRAPH = None
CANDIDATES = ["graph.json", "sunguangrong_kg_v2_explorer.html"]
RAW_URL = "https://raw.githubusercontent.com/psknlr/Master-SGR/claude/tcm-knowledge-graph-app-h709bx/android/app/src/main/assets/web/data/graph.json"

for p in CANDIDATES:
    if os.path.exists(p) and os.path.getsize(p) > 100000:
        GRAPH = p
        print("✔ 使用本地文件：", p, f"({os.path.getsize(p)/1048576:.1f} MB)")
        break

if GRAPH is None:
    print("→ 尝试从 GitHub 拉取…")
    rc = subprocess.call(["wget", "-q", "-O", "graph.json", RAW_URL])
    if rc == 0 and os.path.exists("graph.json") and os.path.getsize("graph.json") > 100000:
        GRAPH = "graph.json"
        print("✔ 已下载 graph.json", f"({os.path.getsize(GRAPH)/1048576:.1f} MB)")
    else:
        if os.path.exists("graph.json"):
            os.remove("graph.json")
        print("✘ 直链不可用（仓库可能为私有）。请在下方选择文件上传：")
        print("   graph.json  或  sunguangrong_kg_v2_explorer.html")
        from google.colab import files
        up = files.upload()
        for name in up:
            if name.endswith((".json", ".html", ".htm")):
                GRAPH = name
                break

assert GRAPH, "未取得图谱数据，请重新运行本单元格并上传文件。"
print("图谱文件：", GRAPH)

## 3　写出源码

三个模块，与 GitHub 仓库 `rag/` 下的文件完全一致：

- `kg_rag.py`　　　图谱索引与检索（双路 BM25 + 图扩展 + 上下文组装 + 子图 SVG）
- `llm_clients.py`　Poe / MiniMax 流式客户端
- `app.py`　　　　Gradio 界面

In [ ]:
%%writefile kg_rag.py
# -*- coding: utf-8 -*-
"""
国医大师孙光荣中医知识图谱 · 检索增强（GraphRAG）引擎
Sun Guangrong TCM Knowledge Graph — retrieval engine for RAG

研发：孙光荣大师弟子田建辉团队 联合 医哲未来人工智能研究院（IMPF-AI Institute）

设计要点
--------
1. 双路召回：实体索引（名称/异名/英文/类型）+ 原文佐证索引（边上的 q 字段）。
   中医术语高度专名化，佐证原文往往就是答案本身，两路互补。
2. 中文用字符二元组做 BM25，不依赖分词器；再叠加「实体名整串命中」的强加权。
3. 图扩展：以命中实体为种子取 1 跳邻域，按「有无原文佐证 / 知识来源 / 种子得分」排序，
   得到一个可解释的子图，而不是一堆散落的文本块。
4. 组装上下文时给每条关系编号 [R1][R2]…，并保留出处编号（如 D3#c0037），
   让模型可以逐条引用、可溯源。
"""

from __future__ import annotations

import html
import json
import math
import os
import re
from collections import defaultdict

import numpy as np

# ---------------------------------------------------------------- 配色（与 App 一致）
ROLE_COLOR = {
    "sun_original":       "#B03A30",   # 朱砂 · 孙光荣原创
    "sun_compiled":       "#B5802C",   # 藤黄 · 孙光荣编纂
    "classical_source":   "#2E5C86",   # 靛青 · 经典引文
    "third_party_clinical": "#4A7A46",  # 竹青 · 他人临床报道
    "general_tcm":        "#7E6F5C",   # 墨灰 · 中医通识
}
ROLE_WEIGHT = {           # 检索排序时的知识来源优先级
    "sun_original": 1.00, "sun_compiled": 0.72, "classical_source": 0.60,
    "third_party_clinical": 0.50, "general_tcm": 0.42,
}

_TOKEN_RE = re.compile(r"[一-鿿]+|[A-Za-z][A-Za-z0-9\-'.]*|\d+")

# 检索默认档位（界面与笔记本的滑杆初值都取这里）
DEFAULT_TOP_SEEDS = 35
DEFAULT_MAX_EDGES = 120
DEFAULT_MAX_NODES = 140
DEFAULT_MAX_CHARS = 9900


def tokenize(text: str):
    """中文切字符二元组，西文按词。二元组对中医专名的召回明显好过单字。"""
    out = []
    for m in _TOKEN_RE.finditer(text or ""):
        s = m.group(0)
        if "一" <= s[0] <= "鿿":
            if len(s) == 1:
                out.append(s)
            else:
                out.extend(s[i:i + 2] for i in range(len(s) - 1))
                if len(s) <= 6:          # 短词整串也作为一个 token，提升精确度
                    out.append(s)
        else:
            out.append(s.lower())
    return out


class BM25:
    """稀疏倒排 + BM25，纯 numpy，无第三方依赖。"""

    def __init__(self, docs, k1=1.5, b=0.75):
        self.k1, self.b = k1, b
        self.N = len(docs)
        self.doc_len = np.zeros(self.N, dtype=np.float32)
        post = defaultdict(lambda: ([], []))
        for i, toks in enumerate(docs):
            self.doc_len[i] = len(toks)
            tf = defaultdict(int)
            for t in toks:
                tf[t] += 1
            for t, c in tf.items():
                ids, cnts = post[t]
                ids.append(i)
                cnts.append(c)
        self.avgdl = float(self.doc_len.mean()) if self.N else 1.0
        self.post = {}
        self.idf = {}
        for t, (ids, cnts) in post.items():
            self.post[t] = (np.asarray(ids, dtype=np.int32), np.asarray(cnts, dtype=np.float32))
            df = len(ids)
            self.idf[t] = math.log(1.0 + (self.N - df + 0.5) / (df + 0.5))

    def score(self, q_tokens):
        acc = np.zeros(self.N, dtype=np.float32)
        seen = set()
        for t in q_tokens:
            if t in seen or t not in self.post:
                continue
            seen.add(t)
            ids, tf = self.post[t]
            denom = tf + self.k1 * (1 - self.b + self.b * self.doc_len[ids] / self.avgdl)
            acc[ids] += self.idf[t] * (tf * (self.k1 + 1) / denom)
        return acc


# ---------------------------------------------------------------- 数据装载
def load_graph(path: str) -> dict:
    """支持两种输入：烘焙好的 graph.json，或原始的 *_explorer.html（自动抽取）。"""
    if not os.path.exists(path):
        raise FileNotFoundError(path)
    if path.lower().endswith((".html", ".htm")):
        src = open(path, encoding="utf-8").read()
        i = src.index("const DATA = ")
        j = src.index("\nconst RC =")
        return json.loads(src[i + len("const DATA = "):j].rstrip().rstrip(";"))
    with open(path, encoding="utf-8") as f:
        return json.load(f)


class Retrieval:
    """一次检索的结果：种子实体、子图实体、关系列表，以及组装好的上下文。"""

    def __init__(self, query, seeds, nodes, edges, kg):
        self.query = query
        self.seeds = seeds          # [(node_id, score)]
        self.nodes = nodes          # [node_id]  已按得分排序
        self.edges = edges          # [(edge_idx, score)]
        self.kg = kg

    def __len__(self):
        return len(self.edges)

    # -------- 表格（供 Gradio / notebook 展示） --------
    def entity_rows(self):
        kg = self.kg
        rows = []
        seed_ids = {i for i, _ in self.seeds}
        for nid in self.nodes:
            n = kg.nodes[nid]
            rows.append([
                "★" if nid in seed_ids else "",
                n["l"],
                kg.type_zh.get(n["t"], n["t"]),
                kg.role_zh.get(n["r"], n["r"]),
                "、".join(n.get("d") or []) or "—",
                int(n.get("g", kg.degree[nid])),
                (n.get("e") or "")[:70],
            ])
        return rows

    ENTITY_HEADERS = ["种子", "实体", "类型", "知识来源", "出处", "关联度", "英文"]

    def triple_rows(self):
        kg = self.kg
        rows = []
        for k, (ei, _) in enumerate(self.edges, 1):
            e = kg.edges[ei]
            rows.append([
                f"R{k}",
                kg.nodes[e["s"]]["l"],
                kg.rel_zh.get(e["y"], e["y"]),
                kg.nodes[e["t"]]["l"],
                (e.get("q") or "").strip() or "—",
                (e.get("c") or e.get("d") or "—"),
                kg.role_zh.get(e.get("r"), e.get("r") or "—"),
                "待审" if e.get("Q") else "",
            ])
        return rows

    TRIPLE_HEADERS = ["编号", "主语", "关系", "宾语", "原文佐证", "出处编号", "知识来源", "标记"]

    # -------- 送入模型的上下文 --------
    def context(self, max_chars=DEFAULT_MAX_CHARS):
        kg = self.kg
        if not self.nodes:
            return "（图谱中未检索到与该问题直接相关的实体。）"

        # 实体与关系分账，防止实体清单把关系挤掉（种子放大到 35 后尤其重要）
        ent_budget = int(max_chars * 0.34)
        buf = ["以下是从《国医大师孙光荣中医知识图谱》检索到的知识片段，"
               f"共 {len(self.nodes)} 个实体、{len(self.edges)} 条关系。\n"]

        buf.append("〖实体〗")
        seed_ids = {i for i, _ in self.seeds}
        used = 0
        for nid in self.nodes:
            n = kg.nodes[nid]
            mark = "★" if nid in seed_ids else "·"
            line = (f"{mark} {n['l']}"
                    f"（{kg.type_zh.get(n['t'], n['t'])}；{kg.role_zh.get(n['r'], n['r'])}）")
            if n.get("e"):
                line += f" [{n['e'][:80]}]"
            docs = n.get("d") or []
            if docs:
                line += "；出处 " + "、".join(f"{d}《{kg.docs.get(d, '')}》" for d in docs[:3])
            if n.get("a"):
                line += "；异名 " + "、".join(n["a"][:4])
            buf.append(line)
            used += len(line)
            if used > ent_budget:
                remain = len(self.nodes) - len(buf) + 2
                if remain > 0:
                    buf.append(f"…（另有 {remain} 个相关实体见下方关系）")
                break

        buf.append("\n〖关系与原文佐证〗")
        # "\n".join 每段还会多一个换行，末尾省略说明也要预留，一并算进来
        used = sum(len(x) for x in buf) + len(buf)
        TAIL = 40
        for k, (ei, _) in enumerate(self.edges, 1):
            e = kg.edges[ei]
            s, t = kg.nodes[e["s"]]["l"], kg.nodes[e["t"]]["l"]
            rel = kg.rel_zh.get(e["y"], e["y"])
            seg = f"[R{k}] {s} —（{rel}）→ {t}"
            q = (e.get("q") or "").strip()
            if q:
                seg += f"\n      佐证：「{q[:220]}」"
            src = e.get("c") or e.get("d")
            if src:
                seg += f"\n      出处：{src}"
                if e.get("d") and kg.docs.get(e["d"]):
                    seg += f"《{kg.docs[e['d']]}》"
                seg += f"（{kg.role_zh.get(e.get('r'), e.get('r') or '')}）"
            if e.get("Q"):
                seg += "  ⚑ 该关系未通过本体校验，仅供参考"
            # 先算再放，保证总字数不越过上限（越界的那条整条不要）
            if used + len(seg) + 1 + TAIL > max_chars and k > 1:
                buf.append(f"…（其余 {len(self.edges) - k + 1} 条关系因篇幅省略）")
                break
            buf.append(seg)
            used += len(seg) + 1

        return "\n".join(buf)

    # -------- 子图 SVG（国风配色，无外部依赖） --------
    #
    # 标签重叠是这类图最伤可读性的问题。这里的做法是：把「圆点 + 标签」当成一个
    # 矩形来布局，力导向收敛后再跑一遍矩形去重叠松弛（PRISM 思路），
    # 于是标签天然不会互相压住。画布尺寸按所有矩形的总面积反推，节点多就自动变大。
    def svg(self, width=None, height=None, theme="paper",
            max_nodes=140, max_labels=44, scroll=True):
        kg = self.kg
        ids = self.nodes[:max_nodes]
        if not ids:
            return ("<div style='padding:28px;text-align:center;color:#7C6C58'>"
                    "暂无检索结果</div>")

        idx = {nid: i for i, nid in enumerate(ids)}
        n = len(ids)
        pairs = []
        for ei, _ in self.edges:
            e = kg.edges[ei]
            a, b = idx.get(e["s"]), idx.get(e["t"])
            if a is not None and b is not None and a != b:
                pairs.append((a, b, kg.rel_zh.get(e["y"], e["y"])))

        bg, fg, sub = (("#F1E6D1", "#241C15", "#7C6C58") if theme == "paper"
                       else ("#131009", "#EDE3D0", "#93866F"))
        seed_ids = {i for i, _ in self.seeds}

        # ---- 1. 谁配标签：种子优先，其余按关联度，给定预算 ----
        order = sorted(range(n), key=lambda i: (ids[i] not in seed_ids,
                                                -int(kg.degree[ids[i]])))
        labeled = set(order[:max_labels])

        # ---- 2. 每个节点的矩形（圆点 + 其下方的标签）----
        def text_w(s, fs):
            return sum(fs if ch > "\u2e7f" else fs * 0.56 for ch in s)

        maxchars = 12 if n <= 40 else (10 if n <= 90 else 8)
        radius = np.empty(n, dtype=np.float64)
        halfw = np.empty(n, dtype=np.float64)
        halfh = np.empty(n, dtype=np.float64)
        labels, fontsz = [], np.zeros(n)
        for i, nid in enumerate(ids):
            is_seed = nid in seed_ids
            deg = int(kg.degree[nid])
            radius[i] = (7.5 if is_seed else 3.6) + min(deg ** 0.35, 3.4)
            if i in labeled:
                raw = kg.nodes[nid]["l"]
                lab = raw[:maxchars] + ("…" if len(raw) > maxchars else "")
                fs = 11.5 if is_seed else 10.0
                labels.append(lab)
                fontsz[i] = fs
                halfw[i] = max(radius[i], text_w(lab, fs) / 2) + 3.5
                halfh[i] = radius[i] + fs + 4.5
            else:
                labels.append("")
                halfw[i] = radius[i] + 2.5
                halfh[i] = radius[i] + 2.5

        # ---- 3. 力导向（Fruchterman-Reingold）----
        box_area = float((4 * halfw * halfh).sum())
        canvas = box_area / 0.20                          # 0.20 ≈ 带标签时实际可达的填充率
        est_w = math.sqrt(canvas * 1.35)                  # 先给力导向一个大致的场地尺度
        rng = np.random.default_rng(11)
        ang = np.arange(n) * 2.399963                      # 黄金角螺旋，起始分布更均匀
        rad = np.sqrt(np.arange(n) + 0.5) * (est_w / (3.2 * math.sqrt(n)))
        pos = np.stack([np.cos(ang) * rad, np.sin(ang) * rad], 1)
        pos += rng.normal(0, 1.5, (n, 2))

        E = (np.array([[a, b] for a, b, _ in pairs], dtype=np.int32)
             if pairs else np.zeros((0, 2), np.int32))
        k = math.sqrt(canvas / max(n, 1)) * 0.62
        temp = est_w * 0.08
        ITERS = 260
        for it in range(ITERS):
            d = pos[:, None, :] - pos[None, :, :]
            dist2 = d[..., 0] ** 2 + d[..., 1] ** 2 + 1e-3
            coef = (k * k) / dist2                          # 斥力 ~ k²/d，方向 d/|d|
            np.fill_diagonal(coef, 0.0)
            disp = np.einsum("ij,ijk->ik", coef, d)
            if len(E):
                dv = pos[E[:, 1]] - pos[E[:, 0]]
                dd = np.hypot(dv[:, 0], dv[:, 1])[:, None] + 1e-6
                att = dv * (dd / k)                         # 引力 ~ d²/k
                np.add.at(disp, E[:, 0], att)
                np.add.at(disp, E[:, 1], -att)
            disp -= pos * 0.035 * (1 + 2 * it / ITERS)       # 重力，后期收紧防飘散
            norm = np.hypot(disp[:, 0], disp[:, 1])[:, None] + 1e-9
            pos += disp / norm * np.minimum(norm, temp)
            temp *= 0.985

        # ---- 4. 定画布：让长宽比贴合版面实际形状，避免一侧大片留白 ----
        pad = 10.0
        legend_h = 46                            # 底部图例条，单独占用不与节点争位
        lo = (pos - np.stack([halfw, halfh], 1)).min(0)
        hi = (pos + np.stack([halfw, halfh], 1)).max(0)
        span = np.maximum(hi - lo, 1e-6)
        if width is None:
            ar = float(np.clip(span[0] / span[1], 0.85, 2.0))
            width = int(np.clip(math.sqrt(canvas * ar), 520, 1380))
        if height is None:
            height = int(np.clip(canvas / max(width, 1), 340, 1100)) + legend_h
        plot_h = height - legend_h

        # ---- 5. 归一到画布并居中 ----
        scale = min((width - 2 * pad) / span[0], (plot_h - 2 * pad) / span[1])
        pos = (pos - lo) * scale
        pos[:, 0] += (width - span[0] * scale) / 2
        pos[:, 1] += (plot_h - span[1] * scale) / 2
        # 盒子不随位置缩放（字号是固定的），所以缩小后要靠松弛重新腾地方

        # ---- 6. 矩形去重叠松弛：标签不再互相压住的关键 ----
        for it in range(180):
            dx = pos[:, 0][:, None] - pos[:, 0][None, :]
            dy = pos[:, 1][:, None] - pos[:, 1][None, :]
            ox = (halfw[:, None] + halfw[None, :]) - np.abs(dx)
            oy = (halfh[:, None] + halfh[None, :]) - np.abs(dy)
            hit = (ox > 0) & (oy > 0)
            np.fill_diagonal(hit, False)
            if not hit.any():
                break
            sx = np.where(dx >= 0, 1.0, -1.0)
            sy = np.where(dy >= 0, 1.0, -1.0)
            along_x = hit & (ox <= oy)                       # 沿穿透较浅的轴推开
            along_y = hit & (ox > oy)
            push = np.stack([
                np.where(along_x, sx * ox * 0.5, 0.0).sum(1),
                np.where(along_y, sy * oy * 0.5, 0.0).sum(1),
            ], 1)
            pos += push * 0.55
            np.clip(pos[:, 0], halfw + pad * 0.4, width - halfw - pad * 0.4, out=pos[:, 0])
            np.clip(pos[:, 1], halfh + pad * 0.4, plot_h - halfh - pad * 0.4, out=pos[:, 1])
        # 提前收敛跳出循环时也要保证在界内
        np.clip(pos[:, 0], halfw + pad * 0.4, width - halfw - pad * 0.4, out=pos[:, 0])
        np.clip(pos[:, 1], halfh + pad * 0.4, plot_h - halfh - pad * 0.4, out=pos[:, 1])

        # ---- 7. 出图 ----
        out = ['<svg xmlns="http://www.w3.org/2000/svg" width="%d" height="%d" '
               'viewBox="0 0 %d %d" style="background:%s;border-radius:4px;display:block">'
               % (width, height, width, height, bg)]

        out.append('<g stroke="%s" stroke-opacity="0.34" stroke-width="1" fill="none">' % sub)
        for a, b, rel in pairs:
            out.append('<line x1="%.1f" y1="%.1f" x2="%.1f" y2="%.1f"><title>%s</title></line>'
                       % (pos[a, 0], pos[a, 1], pos[b, 0], pos[b, 1], html.escape(rel)))
        out.append("</g>")

        for i, nid in enumerate(ids):
            nd = kg.nodes[nid]
            is_seed = nid in seed_ids
            col = ROLE_COLOR.get(nd["r"], "#7E6F5C")
            out.append('<circle cx="%.1f" cy="%.1f" r="%.1f" fill="%s"%s><title>%s</title></circle>'
                       % (pos[i, 0], pos[i, 1], radius[i], col,
                          ' stroke="%s" stroke-width="2.2"' % bg if is_seed else "",
                          html.escape("%s（%s · %s）" % (
                              nd["l"], kg.type_zh.get(nd["t"], nd["t"]),
                              kg.role_zh.get(nd["r"], nd["r"])))))

        # 标签统一画在圆点正下方，配同背景色描边做「光晕」，压住穿过的连线
        for i in range(n):
            if not labels[i]:
                continue
            fs = fontsz[i]
            out.append('<text x="%.1f" y="%.1f" font-size="%.1f" fill="%s" text-anchor="middle" '
                       'stroke="%s" stroke-width="2.6" paint-order="stroke" stroke-linejoin="round" '
                       'font-family="Noto Serif CJK SC,Songti SC,serif"%s>%s</text>'
                       % (pos[i, 0], pos[i, 1] + radius[i] + fs + 0.5, fs, fg, bg,
                          ' font-weight="600"' if ids[i] in seed_ids else "",
                          html.escape(labels[i])))

        # 图例
        legend = [("sun_original", "孙光荣原创"), ("sun_compiled", "孙光荣编纂"),
                  ("classical_source", "经典引文"), ("third_party_clinical", "他人临床报道"),
                  ("general_tcm", "中医通识")]
        lx, ly = 10, plot_h + 20
        out.append('<g font-size="10" font-family="Noto Sans CJK SC,sans-serif" fill="%s" '
                   'opacity="0.88">' % sub)
        cx = lx + 4
        for rk, zh in legend:
            out.append('<circle cx="%.1f" cy="%.1f" r="4" fill="%s"/>' % (cx, ly, ROLE_COLOR[rk]))
            out.append('<text x="%.1f" y="%.1f">%s</text>' % (cx + 9, ly + 3.5, zh))
            cx += 9 + len(zh) * 10 + 20
        out.append('<text x="%d" y="%.1f" font-style="italic">● 描边者为检索种子　共 %d 实体 / %d 关系</text>'
                   % (lx, plot_h + 12, n, len(pairs)))
        out.append("</g></svg>")

        svg = "".join(out)
        if scroll:
            svg = ("<div style='overflow:auto;max-width:100%;max-height:560px;"
                   "border:1px solid rgba(36,28,21,.15);border-radius:4px'>" + svg + "</div>")
        return svg



class KGRag:
    """图谱索引 + 检索。构建一次，反复查询。"""

    def __init__(self, graph_or_path, verbose=True):
        D = load_graph(graph_or_path) if isinstance(graph_or_path, str) else graph_or_path
        self.data = D
        self.nodes = D["nodes"]
        self.edges = D["edges"]
        self.type_zh = {k: v["zh"] for k, v in D["types"].items()}
        self.rel_zh = {k: v["zh"] for k, v in D["rels"].items()}
        self.role_zh = {k: v["zh"] for k, v in D["roles"].items()}
        self.docs = dict(D["docs"])

        n, m = len(self.nodes), len(self.edges)

        # 度与 CSR 邻接（存边号）
        deg = np.zeros(n, dtype=np.int32)
        for e in self.edges:
            deg[e["s"]] += 1
            deg[e["t"]] += 1
        self.degree = deg
        start = np.zeros(n + 1, dtype=np.int64)
        np.cumsum(deg, out=start[1:])
        adj = np.zeros(int(start[-1]), dtype=np.int32)
        cur = start[:-1].copy()
        for k, e in enumerate(self.edges):
            adj[cur[e["s"]]] = k; cur[e["s"]] += 1
            adj[cur[e["t"]]] = k; cur[e["t"]] += 1
        self.adj_start, self.adj = start, adj

        # 实体索引
        node_docs = []
        self.surface = {}                     # 实体名 / 异名 → node id
        for i, nd in enumerate(self.nodes):
            parts = [nd["l"], nd.get("e") or "", self.type_zh.get(nd["t"], "")]
            parts += list(nd.get("a") or [])
            node_docs.append(tokenize(" ".join(parts)))
            for s in [nd["l"]] + list(nd.get("a") or []):
                s = (s or "").strip()
                if len(s) >= 2:
                    self.surface.setdefault(s, i)
        self.node_bm25 = BM25(node_docs)

        # 原文佐证索引（只索引真正带引文的边）
        self.quote_eid = []
        quote_docs = []
        for k, e in enumerate(self.edges):
            q = (e.get("q") or "").strip()
            if not q:
                continue
            self.quote_eid.append(k)
            quote_docs.append(tokenize(
                q + " " + self.rel_zh.get(e["y"], "") + " "
                + self.nodes[e["s"]]["l"] + " " + self.nodes[e["t"]]["l"]))
        self.quote_bm25 = BM25(quote_docs)
        self.quote_eid = np.asarray(self.quote_eid, dtype=np.int32)

        if verbose:
            print(f"✔ 图谱索引就绪：{n:,} 实体 · {m:,} 关系 · "
                  f"{len(self.quote_eid):,} 条带原文佐证 · "
                  f"{len(self.docs)} 部文献 · {len(self.type_zh)} 类实体 · {len(self.rel_zh)} 类关系")

    # ------------------------------------------------------------------ 检索
    def retrieve(self, query, focus_entities=(), top_seeds=DEFAULT_TOP_SEEDS,
                 max_edges=DEFAULT_MAX_EDGES, max_nodes=DEFAULT_MAX_NODES) -> Retrieval:
        q_text = query if not focus_entities else query + " " + " ".join(focus_entities)
        toks = tokenize(q_text)

        node_score = self.node_bm25.score(toks).astype(np.float32)

        # 佐证原文命中 → 回流到两端实体，同时记住直接命中的边
        direct_edges = {}
        if len(self.quote_eid):
            qs = self.quote_bm25.score(toks)
            if qs.max() > 0:
                top_q = np.argpartition(-qs, min(60, len(qs) - 1))[:60]
                top_q = top_q[np.argsort(-qs[top_q])]
                mx = float(qs[top_q[0]]) or 1.0
                for j in top_q:
                    if qs[j] <= 0:
                        break
                    ei = int(self.quote_eid[j])
                    e = self.edges[ei]
                    w = float(qs[j])
                    node_score[e["s"]] += 0.55 * w
                    node_score[e["t"]] += 0.55 * w
                    direct_edges[ei] = 0.9 * w / mx

        # 实体名整串出现在问题里 → 强加权（中医专名往往可精确匹配）
        for surf, nid in self.surface.items():
            if surf in q_text:
                node_score[nid] += 4.0 + 0.9 * len(surf)

        if node_score.max() <= 0:
            return Retrieval(query, [], [], [], self)

        k = min(top_seeds * 4, len(node_score) - 1)
        cand = np.argpartition(-node_score, k)[:k + 1]
        cand = cand[np.argsort(-node_score[cand])]
        # 同名实体（不同文献里的同一概念）只留分最高的一个，把种子名额让给别的概念
        seeds, seen_label = [], set()
        for i in cand:
            if node_score[i] <= 0:
                continue
            lab = self.nodes[int(i)]["l"]
            if lab in seen_label:
                continue
            seen_label.add(lab)
            seeds.append((int(i), float(node_score[i])))
            if len(seeds) >= top_seeds:
                break
        if not seeds:
            return Retrieval(query, [], [], [], self)
        top_score = seeds[0][1] or 1.0

        # 1 跳扩展：按「佐证质量 / 知识来源 / 种子得分 / 对端得分」打分。
        # 枢纽实体（如「失眠」）往往挂着几十条同类型关系，若不加约束，
        # 检索结果会被「见症→失眠」这类重复三元组塞满，因此对每个种子
        # 按关系类型做配额，逼出「主治 / 功效 / 辨证为 / 出典」等不同侧面。
        edge_score = {}
        per_seed_cap = max(6, max_edges // max(1, len(seeds)))
        per_type_cap = 3
        for nid, sc in seeds:
            picked = []
            for p in range(self.adj_start[nid], self.adj_start[nid + 1]):
                ei = int(self.adj[p])
                e = self.edges[ei]
                other = e["t"] if e["s"] == nid else e["s"]
                q = (e.get("q") or "").strip()
                s = (sc / top_score) * 1.0
                # 只有实质性的引文才算佐证；「紫河车9g」这类等同于标签，几乎没有信息量
                s += 0.50 if len(q) >= 10 else (0.16 if len(q) >= 4 else 0.0)
                # 佐证原文若只是把实体名重抄一遍（如「见症→失眠」佐证写作「失眠」），
                # 对回答毫无增量，压到后面去
                if len(q) < 8 and (q in self.nodes[other]["l"] or q in self.nodes[nid]["l"]):
                    s -= 0.40
                s += 0.55 * ROLE_WEIGHT.get(e.get("r"), 0.4)
                s += 0.30 * min(node_score[other] / top_score, 1.0)
                s -= 0.12 * min(math.log1p(self.degree[other]) / 6.0, 1.0)
                if e.get("Q"):
                    s -= 0.25
                picked.append((s, ei, e["y"]))
            picked.sort(key=lambda x: -x[0])

            type_used, taken, spill = defaultdict(int), 0, []
            for s, ei, y in picked:
                if taken >= per_seed_cap:
                    break
                if type_used[y] >= per_type_cap:
                    spill.append((s, ei))
                    continue
                type_used[y] += 1
                taken += 1
                edge_score[ei] = max(edge_score.get(ei, 0.0), s)
            for s, ei in spill[:max(0, per_seed_cap - taken)]:      # 配额没用满再回填
                edge_score[ei] = max(edge_score.get(ei, 0.0), s - 0.3)

        for ei, s in direct_edges.items():                 # 佐证直接命中的边一定保留
            edge_score[ei] = max(edge_score.get(ei, 0.0), s + 1.2)

        # 最终排序：先去掉字面重复的三元组，再按关系类型分桶做轮转取样。
        # 硬配额挡不住「失眠」这种被多个同义种子共同放大的类型——超额的部分
        # 回填时又会把名额抢回去；轮转取样则天然保证各个侧面都有位置，
        # 而在关系类型本来就少的时候又能自动退化成纯按分排序。
        buckets, seen_sig = defaultdict(list), set()
        for ei, sc_e in sorted(edge_score.items(), key=lambda kv: -kv[1]):
            e = self.edges[ei]
            sig = (e["y"], self.nodes[e["s"]]["l"], self.nodes[e["t"]]["l"], (e.get("q") or "")[:24])
            if sig in seen_sig:
                continue
            seen_sig.add(sig)
            buckets[e["y"]].append((ei, sc_e))

        order_types = sorted(buckets, key=lambda y: -buckets[y][0][1])
        bucket_cap = max(3, max_edges // 4)      # 单一关系类型最多占四分之一
        chosen, round_i = [], 0
        while len(chosen) < max_edges and round_i < bucket_cap:
            progressed = False
            for y in order_types:
                if round_i < len(buckets[y]):
                    chosen.append(buckets[y][round_i])
                    progressed = True
                    if len(chosen) >= max_edges:
                        break
            if not progressed:
                break
            round_i += 1
        chosen.sort(key=lambda kv: -kv[1])

        # 子图实体：种子 + 被选中关系的两端，按得分排序
        order, seen = [], set()
        for nid, _ in seeds:
            if nid not in seen:
                seen.add(nid); order.append(nid)
        for ei, _ in chosen:
            for nid in (self.edges[ei]["s"], self.edges[ei]["t"]):
                if nid not in seen:
                    seen.add(nid); order.append(nid)
        order = order[:max_nodes]

        return Retrieval(query, seeds, order, chosen, self)

    def entity_names(self, retrieval, k=3):
        """取本轮核心实体名，用于下一轮的指代消解。"""
        return [self.nodes[i]["l"] for i, _ in retrieval.seeds[:k]]

In [ ]:
%%writefile llm_clients.py
# -*- coding: utf-8 -*-
"""
Poe / MiniMax 对话客户端（流式）
Chat clients for Poe and MiniMax — streaming, dependency-light (requests only).

研发：孙光荣大师弟子田建辉团队 联合 医哲未来人工智能研究院（IMPF-AI Institute）

两家都提供「OpenAI 风格」的接口，但细节不同：
  · Poe      https://api.poe.com/v1/chat/completions    完全 OpenAI 兼容；
             GET /v1/models 无需鉴权即可列出全部可用模型，可据此校正模型名。
  · MiniMax  https://api.minimaxi.chat/v1/text/chatcompletion_v2  （国际站）
             https://api.minimax.chat/v1/text/chatcompletion_v2   （国内站）
             流式分片同为 choices[].delta.content，但错误码放在 base_resp 里，
             且最后一片会把完整文本重复放进 choices[].message.content，需要跳过。
"""

from __future__ import annotations

import json
import time

import requests

POE_BASE = "https://api.poe.com/v1"
MINIMAX_BASE_INTL = "https://api.minimaxi.chat/v1"
MINIMAX_BASE_CN = "https://api.minimax.chat/v1"

# 默认模型。Poe 的模型名是小写点号风格（claude-sonnet-4.6 / minimax-m3）。
DEFAULT_POE_MODEL = "claude-sonnet-5"
DEFAULT_MINIMAX_MODEL = "MiniMax-M3"


class LLMError(RuntimeError):
    pass


def _sse_lines(resp):
    """逐行读 SSE，产出 data: 后面的负载字符串。"""
    for raw in resp.iter_lines(decode_unicode=True):
        if not raw:
            continue
        line = raw.strip()
        if line.startswith("data:"):
            payload = line[5:].strip()
            if payload and payload != "[DONE]":
                yield payload
        elif line == "[DONE]":
            return


class PoeClient:
    """Poe 的 OpenAI 兼容接口。模型名即 Poe 上的 bot 名（小写）。"""

    name = "Poe"

    def __init__(self, api_key: str, model: str = DEFAULT_POE_MODEL,
                 base_url: str = POE_BASE, timeout: int = 180):
        self.api_key = (api_key or "").strip()
        self.model = model or DEFAULT_POE_MODEL
        self.base_url = base_url.rstrip("/")
        self.timeout = timeout

    # ---------- 模型目录 ----------
    def list_models(self):
        """Poe 的模型列表公开可读，不需要 API Key。"""
        r = requests.get(f"{self.base_url}/models", timeout=30)
        r.raise_for_status()
        return [m["id"] for m in r.json().get("data", []) if m.get("id")]

    def resolve_model(self, desired: str | None = None):
        """
        校正模型名：目录里有就直接用；没有则在同族里挑最新的一个。
        Poe 上线新模型的时间可能晚于本项目，这样即使 claude-sonnet-5 尚未上架，
        也会自动退到 claude-sonnet-4.6，而不是直接报 404。
        返回 (实际模型名, 提示信息)。
        """
        desired = (desired or self.model or DEFAULT_POE_MODEL).strip()
        try:
            ids = self.list_models()
        except Exception as e:
            return desired, f"（未能读取 Poe 模型目录：{e}；仍按 {desired} 发起请求）"
        if desired in ids:
            return desired, ""
        low = desired.lower()
        family = low.split("-")[0] + ("-" + low.split("-")[1] if "-" in low else "")
        cands = [i for i in ids if i.lower().startswith(family)]
        if not cands:
            cands = [i for i in ids if i.lower().startswith(low.split("-")[0])]
        if not cands:
            return desired, f"⚠ Poe 目录中没有「{desired}」，也找不到同族替代，请从模型列表中另选。"

        def ver(mid):
            tail = mid.rsplit("-", 1)[-1]
            try:
                return float(tail)
            except ValueError:
                return -1.0

        pick = sorted(cands, key=ver, reverse=True)[0]
        return pick, f"⚠ Poe 目录中暂无「{desired}」，已自动改用同族最新的「{pick}」。"

    # ---------- 对话 ----------
    def stream(self, messages, temperature=0.3, max_tokens=2048):
        if not self.api_key:
            raise LLMError("未配置 Poe API Key（在 https://poe.com/api_key 获取）。")
        body = {
            "model": self.model,
            "messages": messages,
            "stream": True,
            "temperature": temperature,
            "max_tokens": max_tokens,
        }
        try:
            r = requests.post(
                f"{self.base_url}/chat/completions",
                headers={"Authorization": f"Bearer {self.api_key}",
                         "Content-Type": "application/json"},
                json=body, stream=True, timeout=self.timeout)
        except requests.RequestException as e:
            raise LLMError(f"连接 Poe 失败：{e}")

        if r.status_code >= 400:
            raise LLMError(f"Poe 返回 {r.status_code}：{r.text[:400]}")

        for payload in _sse_lines(r):
            try:
                chunk = json.loads(payload)
            except json.JSONDecodeError:
                continue
            if chunk.get("error"):
                raise LLMError(f"Poe 错误：{chunk['error']}")
            for ch in chunk.get("choices") or []:
                piece = (ch.get("delta") or {}).get("content")
                if piece:
                    yield piece


class MiniMaxClient:
    """MiniMax chatcompletion_v2。默认国内站，可切国际站。"""

    name = "MiniMax"

    def __init__(self, api_key: str, model: str = DEFAULT_MINIMAX_MODEL,
                 base_url: str = MINIMAX_BASE_CN, group_id: str = "", timeout: int = 180):
        self.api_key = (api_key or "").strip()
        self.model = model or DEFAULT_MINIMAX_MODEL
        self.base_url = base_url.rstrip("/")
        self.group_id = (group_id or "").strip()
        self.timeout = timeout

    def list_models(self):
        # MiniMax 未提供公开的模型目录接口，给出常用取值供选择
        return ["MiniMax-M3", "MiniMax-M2", "MiniMax-Text-01", "abab6.5s-chat"]

    def resolve_model(self, desired: str | None = None):
        return (desired or self.model or DEFAULT_MINIMAX_MODEL).strip(), ""

    def stream(self, messages, temperature=0.3, max_tokens=2048):
        if not self.api_key:
            raise LLMError("未配置 MiniMax API Key（在 MiniMax 开放平台「账户管理-接口密钥」获取）。")
        url = f"{self.base_url}/text/chatcompletion_v2"
        if self.group_id:
            url += f"?GroupId={self.group_id}"
        body = {
            "model": self.model,
            "messages": messages,
            "stream": True,
            "temperature": max(0.01, temperature),   # MiniMax 不接受 0
            "max_tokens": max_tokens,
        }
        try:
            r = requests.post(
                url,
                headers={"Authorization": f"Bearer {self.api_key}",
                         "Content-Type": "application/json"},
                json=body, stream=True, timeout=self.timeout)
        except requests.RequestException as e:
            raise LLMError(f"连接 MiniMax 失败：{e}")

        if r.status_code >= 400:
            raise LLMError(f"MiniMax 返回 {r.status_code}：{r.text[:400]}")

        streamed = False
        for payload in _sse_lines(r):
            try:
                chunk = json.loads(payload)
            except json.JSONDecodeError:
                continue
            br = chunk.get("base_resp") or {}
            if br.get("status_code"):
                raise LLMError(f"MiniMax 错误 {br.get('status_code')}：{br.get('status_msg')}")
            for ch in chunk.get("choices") or []:
                piece = (ch.get("delta") or {}).get("content")
                if piece:
                    streamed = True
                    yield piece
                elif not streamed:
                    # 非流式兜底：某些情况下只在最后一片给出完整 message
                    full = (ch.get("message") or {}).get("content")
                    if full:
                        streamed = True
                        yield full


def make_client(provider: str, api_key: str, model: str = "", **kw):
    provider = (provider or "").strip().lower()
    if provider.startswith("poe"):
        return PoeClient(api_key, model or DEFAULT_POE_MODEL, **kw)
    if provider.startswith("mini"):
        return MiniMaxClient(api_key, model or DEFAULT_MINIMAX_MODEL, **kw)
    raise LLMError(f"未知的模型提供方：{provider}")


def selftest(client, prompt="用一句话说明「中和」在中医里的含义。"):
    """连通性自检：跑一次极短的请求，返回 (是否成功, 说明)。"""
    t0 = time.time()
    try:
        out = "".join(client.stream(
            [{"role": "user", "content": prompt}], temperature=0.2, max_tokens=120))
    except Exception as e:
        return False, f"✘ {client.name} 自检失败：{e}"
    return True, f"✔ {client.name} / {client.model} 连通（{time.time()-t0:.1f}s）：{out.strip()[:80]}…"

In [ ]:
%%writefile app.py
# -*- coding: utf-8 -*-
"""
国医大师孙光荣中医知识图谱 · 智能问答（Gradio）
Sun Guangrong TCM Knowledge Graph — RAG chat UI

研发：孙光荣大师弟子田建辉团队 联合 医哲未来人工智能研究院（IMPF-AI Institute）

左侧多轮对话，右侧同步展示本轮 RAG 实际检索到的知识图谱数据
（子图 / 实体表 / 三元组与原文佐证 / 送入模型的完整上下文），
答案与证据一屏对照，便于核验，不做黑箱。
"""

from __future__ import annotations

import os
import traceback

import inspect

import gradio as gr

from kg_rag import (DEFAULT_MAX_CHARS, DEFAULT_MAX_EDGES, DEFAULT_TOP_SEEDS, KGRag)
from llm_clients import (DEFAULT_MINIMAX_MODEL, DEFAULT_POE_MODEL, LLMError,
                         MINIMAX_BASE_CN, MINIMAX_BASE_INTL, MiniMaxClient, PoeClient)

SITE_CN = "国内站 api.minimax.chat"
SITE_INTL = "国际站 api.minimaxi.chat"

# 界面初值。Colab 启动格可以整体覆盖（见 build_demo 的 defaults 参数）。
UI_DEFAULTS = {
    "provider": "MiniMax",
    "poe_model": DEFAULT_POE_MODEL,
    "mm_model": DEFAULT_MINIMAX_MODEL,
    "mm_site": SITE_CN,
    "mm_group": "",
    "temperature": 0.3,
    "top_seeds": DEFAULT_TOP_SEEDS,
    "max_edges": DEFAULT_MAX_EDGES,
    "max_ctx": DEFAULT_MAX_CHARS,
}

SYSTEM_PROMPT = """你是「国医大师孙光荣中医知识图谱」的学术问答助手，由孙光荣大师弟子田建辉团队与医哲未来人工智能研究院（IMPF-AI Institute）联合研制。

作答规则：
1. 只依据【图谱证据】作答。证据里没有的内容，明确说「图谱中未见记载」，不得凭常识补充或杜撰。
2. 引用证据时标注编号，如「据 [R3]」；涉及原文时一并给出出处编号（如 D3#c0037）。
3. 区分知识来源：「孙光荣原创」是孙老本人的论断，「孙光荣编纂」是他选编他人内容，「经典引文」出自古籍，
   「他人临床报道」是其他医家的工作。凡属孙老本人观点，请明确说明；不要把他人观点说成孙老的。
4. 标注「⚑ 未通过本体校验」的关系，若要引用须注明其为待审信息。
5. 用中文作答，条理清晰；涉及辨证论治时按「病因病机 — 治则治法 — 方药」的顺序展开。
6. 结尾附一句提醒：本回答仅供学术研究与教学参考，不能替代执业医师的诊断与处方。

【图谱证据】之外的闲聊或与中医无关的问题，可以简短回应并引导回图谱主题。"""

CSS = """
.gradio-container{max-width:1400px!important;
  font-family:"Noto Sans CJK SC","PingFang SC","Microsoft YaHei",system-ui,sans-serif}
#sgr-head{text-align:center;padding:16px 12px 12px;
  background:linear-gradient(180deg,#E3D5B9,#F3E9D6);border:1px solid rgba(36,28,21,.15);
  border-radius:4px;margin-bottom:10px;position:relative}
#sgr-head h1{font-family:"Noto Serif CJK SC","Songti SC",serif;font-size:23px;margin:0 0 2px;
  letter-spacing:.14em;color:#241C15}
#sgr-head .en{font-size:10px;letter-spacing:.2em;color:#7C6C58;text-transform:uppercase}
#sgr-head .team{margin-top:7px;font-family:"Noto Serif CJK SC","Songti SC",serif;
  font-size:12.5px;color:#4C4033;letter-spacing:.04em}
#sgr-head .seal{display:inline-block;background:#A8322A;color:#FBF3E4;border-radius:3px;
  padding:2px 9px;font-family:"Noto Serif CJK SC",serif;font-size:11px;letter-spacing:.2em}
.sgr-panel{background:#EDE1CA;border:1px solid rgba(36,28,21,.15);border-radius:4px;padding:8px}
.sgr-note{font-size:11.5px;color:#7C6C58;line-height:1.8}
footer{display:none!important}
"""

EXAMPLES = [
    "什么是「中和思想」？中和组方的基本原则是什么？",
    "孙光荣治疗脾胃病的学术观点有哪些？",
    "H7N9 禽流感在中医里属于什么范畴，分几个阶段辨证？",
    "带下病的外治法，图谱里有哪些记载？",
    "「三联药组」是什么？举几个孙老常用的药组。",
    "肿瘤的中医治疗思路，孙老怎么讲？",
]


def _c(cls, **kw):
    """
    Gradio 4/5/6 兼容层：只把当前版本签名里存在的参数传下去。
    （Gradio 6 移除了 Chatbot(type=...)、show_copy_button，并把 theme/css 从
      Blocks 挪到了 launch()；Colab 上装到哪个版本不由我们决定，故做签名过滤。）
    """
    allowed = set(inspect.signature(cls.__init__).parameters)
    return cls(**{k: v for k, v in kw.items() if k in allowed})


def _launch_kw(**kw):
    allowed = set(inspect.signature(gr.Blocks.launch).parameters)
    return {k: v for k, v in kw.items() if k in allowed and v is not None}


def _blocks_kw(**kw):
    allowed = set(inspect.signature(gr.Blocks.__init__).parameters)
    return {k: v for k, v in kw.items() if k in allowed}


def _client(provider, poe_key, poe_model, mm_key, mm_model, mm_site, mm_group):
    if provider == "Poe":
        return PoeClient(poe_key or os.environ.get("POE_API_KEY", ""),
                         poe_model or DEFAULT_POE_MODEL)
    base = MINIMAX_BASE_INTL if mm_site == SITE_INTL else MINIMAX_BASE_CN
    return MiniMaxClient(mm_key or os.environ.get("MINIMAX_API_KEY", ""),
                         mm_model or DEFAULT_MINIMAX_MODEL,
                         base_url=base, group_id=mm_group or "")


def build_demo(kg: KGRag, share_note: str = "", show_settings: bool = True,
               defaults: dict | None = None) -> gr.Blocks:
    """
    show_settings=False 时「模型与检索设置」面板整体隐藏（组件仍然存在并生效，
    只是不显示），适合把参数在 Colab 里定好后交付给他人使用的场景。
    """
    D = dict(UI_DEFAULTS)
    if defaults:
        D.update({k: v for k, v in defaults.items() if v is not None and v != ""})

    def check_model(provider, poe_model, mm_model):
        try:
            if provider == "Poe":
                actual, note = PoeClient("", poe_model or DEFAULT_POE_MODEL).resolve_model()
                return f"当前将使用 Poe 模型：**{actual}**　{note}"
            return f"当前将使用 MiniMax 模型：**{mm_model or DEFAULT_MINIMAX_MODEL}**"
        except Exception as e:
            return f"模型校验失败：{e}"

    def respond(message, chat, provider, poe_key, poe_model, mm_key, mm_model,
                mm_site, mm_group, temperature, top_seeds, max_edges, max_ctx, focus):
        message = (message or "").strip()
        if not message:
            yield chat, gr.update(), gr.update(), gr.update(), gr.update(), "请输入问题。", focus
            return

        chat = list(chat or [])
        chat.append({"role": "user", "content": message})

        # ---- 1. 检索：先把图谱证据摆出来，用户不必等模型 ----
        r = kg.retrieve(message, focus_entities=focus or [],
                        top_seeds=int(top_seeds), max_edges=int(max_edges))
        svg = r.svg()
        ents, tris = r.entity_rows(), r.triple_rows()
        context = r.context(max_chars=int(max_ctx))
        stat = (f"本轮检索：种子实体 {len(r.seeds)} · 子图实体 {len(r.nodes)} · "
                f"关系 {len(r.edges)} · 上下文 {len(context)} 字")

        chat.append({"role": "assistant", "content": "正在依据图谱证据作答…"})
        yield chat, svg, ents, tris, context, stat, focus

        # ---- 2. 生成 ----
        try:
            client = _client(provider, poe_key, poe_model, mm_key, mm_model, mm_site, mm_group)
            if provider == "Poe":
                actual, note = client.resolve_model()
                if actual != client.model:
                    client.model = actual
                    stat += f"　|　{note}"

            history_msgs = []
            for m in chat[:-2][-8:]:          # 近 4 轮，控制上下文长度
                if m.get("role") in ("user", "assistant") and m.get("content"):
                    history_msgs.append({"role": m["role"], "content": m["content"]})

            msgs = ([{"role": "system", "content": SYSTEM_PROMPT}] + history_msgs +
                    [{"role": "user", "content": f"{context}\n\n———\n用户问题：{message}"}])

            acc = ""
            for piece in client.stream(msgs, temperature=float(temperature), max_tokens=2048):
                acc += piece
                chat[-1] = {"role": "assistant", "content": acc}
                yield chat, svg, ents, tris, context, stat, focus
            if not acc.strip():
                chat[-1] = {"role": "assistant", "content": "（模型未返回内容，请重试或更换模型）"}
                yield chat, svg, ents, tris, context, stat, focus
        except LLMError as e:
            chat[-1] = {"role": "assistant",
                        "content": f"⚠ {e}\n\n右侧「RAG 知识图谱数据」面板仍展示了本轮检索到的图谱证据，可直接查阅。"}
            yield chat, svg, ents, tris, context, stat, focus
        except Exception:
            chat[-1] = {"role": "assistant", "content": "⚠ 生成出错：\n```\n" + traceback.format_exc()[-900:] + "\n```"}
            yield chat, svg, ents, tris, context, stat, focus
            return

        # ---- 3. 记住本轮核心实体，供下一轮指代消解（「它」「这个方」…）----
        yield chat, svg, ents, tris, context, stat, kg.entity_names(r, 3)

    theme = gr.themes.Base(
        primary_hue=gr.themes.colors.red, neutral_hue=gr.themes.colors.stone,
        font=["Noto Sans CJK SC", "PingFang SC", "sans-serif"],
    )

    with gr.Blocks(**_blocks_kw(title="国医大师孙光荣中医知识图谱 · 智能问答",
                               theme=theme, css=CSS)) as demo:
        gr.HTML(f"""
        <div id="sgr-head">
          <span class="seal">中和</span>
          <h1>国医大师孙光荣中医知识图谱 · 智能问答</h1>
          <div class="en">Sun Guangrong TCM Knowledge Graph — Graph-RAG Assistant</div>
          <div class="team">孙光荣大师弟子 <b>田建辉</b> 团队　联合研发　<b>医哲未来人工智能研究院</b>（IMPF-AI Institute）</div>
          <div class="en" style="margin-top:5px">
            {kg.degree.size:,} 实体 · {len(kg.edges):,} 关系 · 8 部文献 · 检索增强问答{share_note}</div>
        </div>""")

        focus = gr.State([])

        with gr.Row():
            # ------------------ 左：多轮对话 ------------------
            with gr.Column(scale=9):
                chatbot = _c(gr.Chatbot, type="messages", height=560, label="问答",
                             avatar_images=(None, None), show_copy_button=True)
                with gr.Row():
                    box = _c(gr.Textbox, placeholder="请就孙光荣学术思想、辨证、方药、医案提问…",
                             scale=8, lines=2, max_lines=6, show_label=False, autofocus=True)
                    send = gr.Button("问 道", variant="primary", scale=1, min_width=88)
                with gr.Row():
                    clear = gr.Button("清空对话", size="sm", scale=1)
                    gr.Markdown("", scale=4)
                status = gr.Markdown("", elem_classes="sgr-note")
                gr.Examples(EXAMPLES, inputs=box, label="示例问题")

            # ------------------ 右：RAG 知识图谱数据 ------------------
            with gr.Column(scale=11):
                gr.Markdown("### RAG 知识图谱数据\n"
                            "<span class='sgr-note'>本轮回答实际依据的图谱证据，与左侧答案一一对应。</span>")
                with gr.Tab("检索子图"):
                    sub_svg = _c(gr.HTML,
                                 value="<div style='padding:28px;text-align:center;color:#7C6C58'>"
                                       "提问后在此展示检索到的子图</div>", label="检索子图")
                with gr.Tab("三元组与佐证"):
                    tri_df = _c(gr.Dataframe, headers=r_headers_tri(), interactive=False,
                                wrap=True, max_height=520, show_label=False,
                                column_widths=["7%", "17%", "11%", "17%", "31%", "12%", "10%", "7%"])
                with gr.Tab("实体"):
                    ent_df = _c(gr.Dataframe, headers=r_headers_ent(), interactive=False,
                                wrap=True, max_height=520, show_label=False,
                                column_widths=["6%", "26%", "13%", "14%", "13%", "9%", "22%"])
                with gr.Tab("送入模型的上下文"):
                    ctx_box = _c(gr.Textbox, lines=22, max_lines=22, show_label=False,
                                 show_copy_button=True, interactive=False)

        # ------------------ 设置（可整体隐藏） ------------------
        with _c(gr.Accordion, label="模型与检索设置", open=False, visible=show_settings):
            with gr.Row():
                provider = gr.Radio(["MiniMax", "Poe"], value=D["provider"],
                                    label="模型提供方", scale=2)
                temperature = gr.Slider(0.0, 1.0, value=D["temperature"], step=0.05,
                                        label="温度", scale=3)
            with gr.Row():
                with gr.Column():
                    gr.Markdown("**MiniMax**　密钥在开放平台「账户管理 - 接口密钥」获取")
                    mm_model = gr.Textbox(D["mm_model"], label="MiniMax 模型")
                    mm_key = _c(gr.Textbox, value="", label="MiniMax API Key", type="password",
                                placeholder="留空则用环境变量 MINIMAX_API_KEY")
                    mm_site = gr.Radio([SITE_CN, SITE_INTL], value=D["mm_site"], label="接入站点")
                    mm_group = gr.Textbox(D["mm_group"], label="GroupId（可选）")
                with gr.Column():
                    gr.Markdown("**Poe**　密钥在 https://poe.com/api_key 获取")
                    poe_model = _c(gr.Textbox, value=D["poe_model"], label="Poe 模型",
                                   info="Poe 上尚未上架时会自动退到同族最新版")
                    poe_key = _c(gr.Textbox, value="", label="Poe API Key", type="password",
                                 placeholder="留空则用环境变量 POE_API_KEY")
            with gr.Row():
                top_seeds = gr.Slider(3, 60, value=D["top_seeds"], step=1, label="种子实体数")
                max_edges = gr.Slider(12, 300, value=D["max_edges"], step=4, label="检索关系数")
                max_ctx = gr.Slider(2000, 20000, value=D["max_ctx"], step=100,
                                    label="上下文字数上限")
            check_btn = gr.Button("校验当前模型名", size="sm")
            check_out = gr.Markdown("")
            check_btn.click(check_model, [provider, poe_model, mm_model], check_out)

        gr.Markdown(
            "<div class='sgr-note' style='text-align:center;margin-top:10px'>"
            "本应用为中医药学术研究与教学参考工具，所载内容由文献自动抽取并经抽样审校，"
            "<b>不能替代执业医师的诊断与处方</b>，请勿据此自行用药。<br>"
            "孙光荣大师弟子田建辉团队 × 医哲未来人工智能研究院（IMPF-AI Institute）"
            "</div>")

        ins = [box, chatbot, provider, poe_key, poe_model, mm_key, mm_model,
               mm_site, mm_group, temperature, top_seeds, max_edges, max_ctx, focus]
        outs = [chatbot, sub_svg, ent_df, tri_df, ctx_box, status, focus]
        send.click(respond, ins, outs).then(lambda: "", None, box)
        box.submit(respond, ins, outs).then(lambda: "", None, box)
        clear.click(lambda: ([], [], ""), None, [chatbot, focus, status])

    return demo


def r_headers_ent():
    from kg_rag import Retrieval
    return Retrieval.ENTITY_HEADERS


def r_headers_tri():
    from kg_rag import Retrieval
    return Retrieval.TRIPLE_HEADERS


def launch(demo, share=False, port=7860, auth=None, quiet=False):
    """统一的启动入口：Colab 与本地共用，屏蔽 Gradio 版本差异。"""
    theme = gr.themes.Base(primary_hue=gr.themes.colors.red,
                           neutral_hue=gr.themes.colors.stone)
    return demo.queue(default_concurrency_limit=4).launch(
        **_launch_kw(share=share, server_port=port, server_name="0.0.0.0",
                     auth=auth, show_api=False, quiet=quiet, theme=theme, css=CSS,
                     inline=False, debug=False))


def main():
    import argparse
    ap = argparse.ArgumentParser(description="孙光荣中医知识图谱 · RAG 问答")
    ap.add_argument("--graph", default="graph.json", help="graph.json 或原始 explorer.html")
    ap.add_argument("--share", action="store_true", help="生成公网可访问链接")
    ap.add_argument("--port", type=int, default=7860)
    ap.add_argument("--user", default="", help="可选：访问用户名")
    ap.add_argument("--password", default="", help="可选：访问密码")
    ap.add_argument("--hide-settings", action="store_true", help="隐藏「模型与检索设置」面板")
    a = ap.parse_args()

    kg = KGRag(a.graph)
    demo = build_demo(kg, show_settings=not a.hide_settings)
    launch(demo, share=a.share, port=a.port,
           auth=(a.user, a.password) if a.user and a.password else None)


if __name__ == "__main__":
    main()

## 4　构建检索索引

In [ ]:
#@title 构建索引（约 5 秒）
import sys, importlib
sys.path.insert(0, ".")
for m in ("kg_rag", "llm_clients", "app"):
    if m in sys.modules:
        importlib.reload(sys.modules[m])

from kg_rag import KGRag
kg = KGRag(GRAPH)

## 5　RAG 知识图谱数据预览　🔍

**这一格单独展示 RAG 到底检索到了什么**——改上面的问题重跑即可，不需要 API Key。
（同样的内容也会实时出现在第 8 格问答界面的右侧面板里。）

In [ ]:
#@title RAG 检索到的知识图谱数据 { display-mode: "form" }
问题 = "什么是中和思想？中和组方的基本原则是什么？"  #@param {type:"string"}
种子实体数 = 35   #@param {type:"slider", min:3, max:60, step:1}
检索关系数 = 120  #@param {type:"slider", min:12, max:300, step:4}
上下文字数上限 = 9900  #@param {type:"slider", min:2000, max:20000, step:100}

import pandas as pd
from IPython.display import HTML, display

r = kg.retrieve(问题, top_seeds=int(种子实体数), max_edges=int(检索关系数))
ctx = r.context(max_chars=int(上下文字数上限))

display(HTML(
    f"<div style='font-family:Noto Serif CJK SC,Songti SC,serif;font-size:15px;"
    f"padding:8px 0;color:#241C15'><b>「{问题}」</b></div>"
    f"<div style='color:#7C6C58;font-size:12.5px;padding-bottom:8px'>"
    f"种子实体 {len(r.seeds)} · 子图实体 {len(r.nodes)} · 关系 {len(r.edges)} · "
    f"上下文 {len(ctx)} 字</div>"))

display(HTML("<h4>① 检索子图</h4>" + r.svg()))   # 画布尺寸随实体数自适应

df_t = pd.DataFrame(r.triple_rows(), columns=r.TRIPLE_HEADERS)
df_e = pd.DataFrame(r.entity_rows(), columns=r.ENTITY_HEADERS)

display(HTML("<h4>② 三元组与原文佐证</h4>"))
display(df_t.style.hide(axis="index").set_properties(
    **{"text-align": "left", "font-size": "12.5px", "white-space": "pre-wrap"}))

display(HTML("<h4>③ 实体（★ 为检索种子）</h4>"))
display(df_e.style.hide(axis="index").set_properties(
    **{"text-align": "left", "font-size": "12.5px"}))

print("\n④ 送入模型的上下文（前 1500 字）\n" + "─" * 60)
print(ctx[:1500])
print("─" * 60, f"\n共 {len(ctx)} 字")

## 6　配置模型 API

支持两家，二选一或都填：

| | 默认模型 | 密钥获取 |
|---|---|---|
| **MiniMax**（默认） | `MiniMax-M3`　国内站 `api.minimax.chat` | MiniMax 开放平台 → 账户管理 → 接口密钥 |
| **Poe** | `claude-sonnet-5` | <https://poe.com/api_key> |

推荐把密钥存进 Colab 左侧 🔑 **Secrets**（名字用 `POE_API_KEY` / `MINIMAX_API_KEY`），
这样密钥不会留在笔记本里。没存 Secrets 时会提示手动输入，直接回车可跳过。

> Poe 的模型目录会实时校验：若 `claude-sonnet-5` 尚未上架，会自动改用同族最新版本
> （目前是 `claude-sonnet-4.6`）并给出提示。

In [ ]:
#@title 填写 / 读取 API 密钥并自检
import os, getpass

def get_key(name):
    try:
        from google.colab import userdata
        v = userdata.get(name)
        if v:
            print(f"  {name}: 已从 Colab Secrets 读取")
            return v.strip()
    except Exception:
        pass
    v = os.environ.get(name, "")
    if v:
        print(f"  {name}: 已从环境变量读取")
        return v.strip()
    try:
        v = getpass.getpass(f"  请粘贴 {name}（不用可直接回车）：")
    except Exception:
        v = ""
    return (v or "").strip()

POE_API_KEY = get_key("POE_API_KEY")
MINIMAX_API_KEY = get_key("MINIMAX_API_KEY")
os.environ["POE_API_KEY"] = POE_API_KEY
os.environ["MINIMAX_API_KEY"] = MINIMAX_API_KEY

from llm_clients import (DEFAULT_MINIMAX_MODEL, DEFAULT_POE_MODEL,
                         MiniMaxClient, PoeClient, selftest)

print("\n— Poe 模型目录校验 —")
actual, note = PoeClient("", DEFAULT_POE_MODEL).resolve_model()
print(f"  期望 {DEFAULT_POE_MODEL} → 实际 {actual}　{note}")

print("\n— 连通性自检 —")
if POE_API_KEY:
    print(" ", selftest(PoeClient(POE_API_KEY, actual))[1])
else:
    print("  · 未提供 Poe 密钥，跳过")
if MINIMAX_API_KEY:
    print(" ", selftest(MiniMaxClient(MINIMAX_API_KEY, DEFAULT_MINIMAX_MODEL))[1])
else:
    print("  · 未提供 MiniMax 密钥，跳过")

if not (POE_API_KEY or MINIMAX_API_KEY):
    print("\n⚠ 两家都没配密钥：问答界面仍可启动，RAG 检索面板照常工作，"
          "但生成回答时会提示缺少密钥。")

## 7　启动问答服务，生成公网链接　🌐

运行后会打印一条 `https://xxxxxxxx.gradio.live` 链接，
**手机、其他电脑都能直接打开**，支持多轮对话，有效期 72 小时。

- 想给链接加口令：把 `访问用户名` / `访问密码` 填上。
- 想长期在线：把 `rag/` 目录部署到 Hugging Face Spaces 或自己的服务器（见仓库 README）。

In [ ]:
#@title 启动 Gradio（生成公网链接） { display-mode: "form" }
#@markdown ### 默认模型
模型提供方 = "MiniMax"  #@param ["MiniMax", "Poe"]
MiniMax模型 = "MiniMax-M3"  #@param {type:"string"}
MiniMax站点 = "国内站 api.minimax.chat"  #@param ["国内站 api.minimax.chat", "国际站 api.minimaxi.chat"]
MiniMax_GroupId = ""  #@param {type:"string"}
Poe模型 = "claude-sonnet-5"  #@param {type:"string"}
温度 = 0.3  #@param {type:"slider", min:0, max:1, step:0.05}

#@markdown ### 默认检索档位
种子实体数 = 35   #@param {type:"slider", min:3, max:60, step:1}
检索关系数 = 120  #@param {type:"slider", min:12, max:300, step:4}
上下文字数上限 = 9900  #@param {type:"slider", min:2000, max:20000, step:100}

#@markdown ### 界面
#@markdown 取消勾选则**隐藏**网页上的「模型与检索设置」面板 ——
#@markdown 参数仍按上面设定生效，只是使用者改不了。
显示参数设置面板 = False  #@param {type:"boolean"}
访问用户名 = ""  #@param {type:"string"}
访问密码 = ""    #@param {type:"string"}

import importlib
import app as sgr_app
importlib.reload(sgr_app)

demo = sgr_app.build_demo(
    kg, share_note=" · Colab 在线版",
    show_settings=显示参数设置面板,
    defaults=dict(provider=模型提供方, mm_model=MiniMax模型, mm_site=MiniMax站点,
                  mm_group=MiniMax_GroupId, poe_model=Poe模型, temperature=温度,
                  top_seeds=种子实体数, max_edges=检索关系数, max_ctx=上下文字数上限),
)
auth = (访问用户名, 访问密码) if (访问用户名 and 访问密码) else None

res = sgr_app.launch(demo, share=True, port=7860, auth=auth)
try:
    print("公网链接：", res[2] or "(未生成，请检查 Colab 网络)")
    print("本地链接：", res[1])
except Exception:
    pass

print("参数设置面板：", "显示" if 显示参数设置面板 else "已隐藏")
print("停止服务：demo.close()")

---

## 常见问题

**公网链接打不开 / 没有生成？**　Colab 偶尔无法连上 gradio 的隧道服务，重跑第 8 格即可；
链接有效期 72 小时，会话断开后需要重跑。

**回答说「图谱中未见记载」？**　这是设计如此——系统只依据检索到的图谱证据作答，
不会用通用常识编补。可以在第 6 格看看究竟检索到了什么，或调大「检索关系数」再试。

**怎么换模型？**　界面里「模型与检索设置 → Poe 模型 / MiniMax 模型」直接改。
Poe 的可用模型名可以这样列出：

```python
from llm_clients import PoeClient
ids = PoeClient("").list_models()
print([i for i in ids if "claude" in i or "minimax" in i])
```

**想停止服务？**　`demo.close()`，或直接中断该单元格。

---

<div align="center">

**孙光荣大师弟子田建辉团队** × **医哲未来人工智能研究院（IMPF-AI Institute）**

承国医之道 · 启智能之钥

</div>